# Clase 05 — Sobreajuste, predicción raster y clasificación supervisada

Este notebook reproduce la práctica de la Clase 05 con los datos locales de `SINCHI.gdb` y conecta el flujo con lo aprendido en Clase 04 sobre biomasa, variables explicativas, sobreajuste e índices de teledetección.

**Pregunta guía:** ¿cómo pasamos de polígonos de referencia bosque/no bosque y una imagen multibanda a un clasificador raster evaluable?

**Qué produce:** inventarios, tablas de clases, entrenamiento/fallback de Random Trees, raster clasificado, matriz de confusión sobre la zona de entrenamiento, mapas PNG y una interpretación metodológica.

**Docente:** Fabian Cetina · **Fecha:** 22/09/2026

## Objetivos
- Conservar la secuencia SINCHI: muestras, entrenamiento, clasificación y evaluación.
- Distinguir ajuste aparente, validación espacial y complementos metodológicos.
- Interpretar balance, hiperparámetros y consenso.

**Estado:** adaptación no ejecutada. Las salidas históricas permanecen en el notebook original de Clase 17 (01/07/2026). Sus métricas afectadas por interpolación, recodificación o alineación no validan el código corregido.


## 1. Preparación del entorno

**Qué:** configuramos rutas, salidas y extensiones de ArcGIS Pro.  
**Por qué:** los flujos raster dependen de rutas estables, licencias y geodatabases de salida.  
**Para qué:** dejar una ejecución reproducible, sin depender de pasos manuales en la interfaz.

La primera celda configura las rutas; para entradas externas cambie DATA_DIR y SINCHI_GDB allí. Cada reinicio reserva un nombre de salida nuevo. Los datos originales nunca se usan como destino.


In [ ]:
# Cambie las rutas por carpetas externas si lo necesita; entradas solo de lectura.
from pathlib import Path
from uuid import uuid4
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "99 - Recursos").is_dir()), Path.cwd())
DATA_DIR = ROOT / "99 - Recursos/datos/clase_05"
OUTPUT_DIR = ROOT / "99 - Recursos/salidas_clase_05/practica_01" / ("ejecucion_" + uuid4().hex[:12])
SINCHI_GDB = DATA_DIR / "SINCHI.gdb"


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
import os
from pathlib import Path
import json
import math
import arcpy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = DATA_DIR
OUT = OUTPUT_DIR
if not SINCHI_GDB.is_dir():
    raise FileNotFoundError(SINCHI_GDB)
if SINCHI_GDB.resolve() in OUT.resolve().parents or OUT.resolve() == SINCHI_GDB.resolve():
    raise ValueError("Las salidas no pueden estar dentro de la entrada")
OUT.mkdir(parents=True, exist_ok=False)
OUT_GDB = OUT / "clase_05_sinchi_rf.gdb"
if not arcpy.Exists(str(OUT_GDB)):
    arcpy.management.CreateFileGDB(str(OUT), OUT_GDB.name)

arcpy.env.overwriteOutput = False
arcpy.env.workspace = str(SINCHI_GDB)

# Disponibilidad no equivale a prestamo efectivo de licencia.
for extension in ("ImageAnalyst", "Spatial"):
    if arcpy.CheckOutExtension(extension) != "CheckedOut":
        raise RuntimeError(f"No se obtuvo la licencia {extension}")

paths = {
    "sinchi_gdb": str(SINCHI_GDB),
    "out_dir": str(OUT),
    "out_gdb": str(OUT_GDB),
}
paths


## 2. Inventario de datos

**Qué:** listamos entidades, rasters y campos clave.  
**Por qué:** antes de modelar hay que saber qué datos existen, en qué sistema de referencia están y qué papel cumple cada capa.  
**Para qué:** evitar entrenar un modelo con insumos equivocados o sin documentar.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
rows = []
arcpy.env.workspace = str(SINCHI_GDB)
for fc in arcpy.ListFeatureClasses() or []:
    d = arcpy.Describe(fc)
    rows.append({
        "tipo": "feature_class",
        "nombre": fc,
        "geometria": getattr(d, "shapeType", ""),
        "conteo": int(arcpy.management.GetCount(fc)[0]),
        "sr": d.spatialReference.name,
        "campos": ", ".join([f.name for f in arcpy.ListFields(fc)[:12]]),
    })
for r in arcpy.ListRasters() or []:
    d = arcpy.Describe(r)
    rows.append({
        "tipo": "raster",
        "nombre": r,
        "geometria": f"{getattr(d, 'bandCount', '')} banda(s)",
        "conteo": "",
        "sr": d.spatialReference.name,
        "campos": f"extent=({d.extent.XMin:.0f}, {d.extent.YMin:.0f}, {d.extent.XMax:.0f}, {d.extent.YMax:.0f})",
    })

inventario = pd.DataFrame(rows)
inventario.to_csv(OUT / "clase_05_inventario_sinchi.csv", index=False, encoding="utf-8-sig")
inventario


## Nota metodológica: `Band_5` no significa automáticamente Sentinel-2 `B5`

La geodatabase muestra los rasters compuestos como bandas genéricas de ArcGIS:

- `Raster_compuesto/Band_1` ... `Raster_compuesto/Band_5`;
- `Raster_compuesto_Clip/Band_1` ... `Raster_compuesto_Clip/Band_5`;
- `Raster_test/Band_1` ... `Raster_test/Band_5`.

Esos nombres son **posicionales**. `Band_5` significa “quinta banda del raster compuesto”, no necesariamente la banda espectral Sentinel-2 `B5`.

En cambio, la capa de puntos `SINCHI_Guaviare_RandomPoints` ya trae campos tabulares con nombres interpretativos (`B2`, `B3`, `B4`, `B8`, `NDVI`). En este notebook:

- cuando se habla del **raster compuesto**, se usan los nombres `Band_1` a `Band_5`;
- cuando se habla de la **tabla de puntos**, se usan los campos disponibles `B2`, `B3`, `B4`, `B8`, `NDVI`;
- no se asume que `Band_5 = Sentinel B5`, ni que `Band_5 = B8`, sin metadatos externos que documenten el orden de composición.

La clasificación se entrena con el raster compuesto completo y depende del **orden de bandas del compuesto**. Por eso un `.ecd` solo debe reutilizarse con un raster que tenga exactamente el mismo orden y significado de bandas.


In [ ]:
band_rows = []
# Registro explícito a partir de lo observado en ArcGIS Pro: los compuestos tienen 5 bandas genéricas.
for raster_name, n_bands in [("Raster_compuesto", 5), ("Raster_compuesto_Clip", 5), ("Raster_test", 5), ("NDVI_20230823_10m", 1)]:
    for band_idx in range(1, n_bands + 1):
        band_rows.append({
            "raster": raster_name,
            "banda_arcgis": f"Band_{band_idx}",
            "nota": "nombre posicional dentro del raster; no equivale automáticamente a una banda Sentinel específica",
        })

band_metadata = pd.DataFrame(band_rows)
band_metadata.to_csv(OUT / "clase_05_metadatos_bandas_raster.csv", index=False, encoding="utf-8-sig")
band_metadata


## 3. Lectura de clases de referencia

**Qué:** revisamos las clases bosque/no bosque en polígonos y puntos de entrenamiento.  
**Por qué:** en clasificación supervisada el modelo aprende desde etiquetas; si las clases están desbalanceadas o mal codificadas, el resultado se sesga.  
**Para qué:** comprobar que tenemos una muestra balanceada antes de entrenar.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
poly = str(SINCHI_GDB / "SINCHI_Guaviare")
pts = str(SINCHI_GDB / "SINCHI_Guaviare_RandomPoints")

def table_counts(fc, field):
    data=[]
    with arcpy.da.SearchCursor(fc, [field]) as cur:
        for (v,) in cur:
            data.append(v)
    return pd.Series(data, name=field).value_counts(dropna=False).rename_axis(field).reset_index(name="conteo")

conteo_poligonos = table_counts(poly, "descripcio")
conteo_puntos = table_counts(pts, "classname")
conteo_puntos["porcentaje"] = (conteo_puntos["conteo"] / conteo_puntos["conteo"].sum() * 100).round(2)
conteo_poligonos.to_csv(OUT / "clase_05_conteo_poligonos.csv", index=False, encoding="utf-8-sig")
conteo_puntos.to_csv(OUT / "clase_05_conteo_puntos_entrenamiento.csv", index=False, encoding="utf-8-sig")
conteo_puntos


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(conteo_puntos["classname"].astype(str), conteo_puntos["conteo"], color=["#2e7d32", "#f9a825"][:len(conteo_puntos)])
ax.set_title("Clase 05 — puntos de entrenamiento por clase")
ax.set_ylabel("Número de puntos")
ax.set_xlabel("Clase")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=10)
plt.tight_layout()
fig_path = OUT / "clase_05_balance_muestras.png"
plt.savefig(fig_path, dpi=160)
plt.show()
fig_path

# Grafico nativo ArcGIS: el balance no implica cobertura espacial suficiente.
balance_chart = arcpy.charts.Bar(x="classname", aggregation="COUNT", dataSource=pts,
    title="Puntos por clase", xTitle="Clase", yTitle="Numero de puntos")
balance_chart.exportToSVG(str(OUT / "balance_muestras_arcgis.svg"))
display(balance_chart)


## 4. Mapa exploratorio de puntos de entrenamiento

**Qué:** dibujamos la distribución espacial de las muestras.  
**Por qué:** una muestra balanceada en conteos puede seguir siendo débil si está concentrada espacialmente.  
**Para qué:** revisar si el entrenamiento cubre razonablemente el área de aprendizaje.


In [ ]:
records=[]
# Estos son campos tabulares existentes en los puntos, no nombres de bandas del raster compuesto.
with arcpy.da.SearchCursor(pts, ["SHAPE@XY", "classname", "classvalue", "NDVI", "B2", "B3", "B4", "B8"]) as cur:
    for xy, name, val, ndvi, b2, b3, b4, b8 in cur:
        records.append({"x": xy[0], "y": xy[1], "classname": name, "classvalue": val, "NDVI": ndvi, "B2": b2, "B3": b3, "B4": b4, "B8": b8})
puntos = pd.DataFrame(records)
puntos.to_csv(OUT / "clase_05_puntos_entrenamiento_muestra.csv", index=False, encoding="utf-8-sig")

colors = {name: c for name, c in zip(sorted(puntos["classname"].dropna().unique()), ["#2e7d32", "#f9a825", "#1565c0"])}
fig, ax = plt.subplots(figsize=(7,7))
for name, grp in puntos.groupby("classname"):
    ax.scatter(grp["x"], grp["y"], s=5, alpha=0.65, label=name, color=colors.get(name, "#555"))
ax.set_title("Distribución de puntos de entrenamiento")
ax.set_xlabel("Longitud / X")
ax.set_ylabel("Latitud / Y")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
fig_path = OUT / "clase_05_mapa_puntos_entrenamiento.png"
plt.savefig(fig_path, dpi=170)
plt.show()
puntos.head()


## 5. Campos espectrales ya extraídos en los puntos

**Qué:** comparamos los campos tabulares `NDVI`, `B2`, `B3`, `B4` y `B8` que ya vienen en `SINCHI_Guaviare_RandomPoints`.  
**Por qué:** estos campos ayudan a auditar la separabilidad espectral de las clases, pero no son los nombres visibles de bandas del raster compuesto.  
**Para qué:** entender la muestra de entrenamiento sin confundir campos de puntos con `Band_1`–`Band_5` del raster.


In [ ]:
# Resumen de campos de puntos: B8 es campo tabular; el raster compuesto visible usa Band_1-Band_5.
summary = puntos.groupby("classname")[["NDVI","B2","B3","B4","B8"]].agg(["count","mean","std","min","max"]).round(3)
summary.to_csv(OUT / "clase_05_resumen_espectral_por_clase.csv", encoding="utf-8-sig")
summary


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
fig, axes = plt.subplots(1, 2, figsize=(11,4))
for i, var in enumerate(["NDVI", "B8"]):
    data = [grp[var].dropna().values for _, grp in puntos.groupby("classname")]
    labels = [name for name, _ in puntos.groupby("classname")]
    axes[i].boxplot(data, labels=labels, patch_artist=True)
    titulo = f"Distribución de {var} por clase"
    if var == "B8":
        titulo += " (campo de puntos, no Band_5)"
    axes[i].set_title(titulo)
    axes[i].set_ylabel(var)
    axes[i].grid(axis="y", alpha=0.25)
plt.tight_layout()
fig_path = OUT / "clase_05_distribucion_espectral_clases.png"
plt.savefig(fig_path, dpi=160)
plt.show()
fig_path


## 6. Entrenamiento Random Trees

**Qué:** entrenamos un clasificador Random Trees con el raster compuesto (`Band_1`–`Band_5`) y los puntos etiquetados.  
**Por qué:** Random Trees aprende reglas no lineales a partir de las bandas del compuesto, y genera un archivo `.ecd` reutilizable solo si se conserva el mismo orden de bandas.  
**Para qué:** aplicar el modelo a un raster de prueba compatible y producir una clasificación bosque/no bosque.

Si falla el entrenamiento, la ejecución se detiene: no se usa un `.ecd` histórico sin comprobar su compatibilidad espectral.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
raster_train = str(SINCHI_GDB / "Raster_compuesto_Clip")
raster_test = str(SINCHI_GDB / "Raster_test")
training_fc = str(SINCHI_GDB / "SINCHI_Guaviare_RandomPoints")
out_ecd = OUT / "clase_05_sinchi_random_trees.ecd"


train_log = []
used_ecd = out_ecd
try:
    result = arcpy.ia.TrainRandomTreesClassifier(
        raster_train,
        training_fc,
        str(out_ecd),
        "#",
        80,      # árboles: suficiente para práctica, no producto final
        30,      # profundidad máxima: reproduce una configuración flexible
        1000,    # muestras máximas por clase: balanceado según la clase
        "#"
    )
    train_log.append({"paso": "entrenamiento", "estado": "ok", "detalle": str(out_ecd)})
except Exception as exc:
    raise RuntimeError("Entrenamiento fallido; no se reutiliza un modelo no verificado") from exc

pd.DataFrame(train_log).to_csv(OUT / "clase_05_entrenamiento_log.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(train_log)


## 7. Clasificación raster

**Qué:** aplicamos el `.ecd` a la zona de entrenamiento y a un raster de prueba.  
**Por qué:** la salida útil del clasificador es un raster categórico; cada celda recibe una clase.  
**Para qué:** obtener un mapa bosque/no bosque y evaluar la consistencia sobre puntos conocidos.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
classified_train = str(OUT_GDB / "clase_05_clasificacion_train")
classified_test = str(OUT_GDB / "clase_05_clasificacion_test")

cls_train = arcpy.ia.ClassifyRaster(raster_train, str(used_ecd))
cls_train.save(classified_train)
# Solo cambiar tras verificar metadatos de composicion, nunca por nombres Band_N.
IDENTIDAD_BANDAS_VERIFICADA = False
if not IDENTIDAD_BANDAS_VERIFICADA:
    raise RuntimeError("Falta comprobar mismo significado y orden de bandas de Raster_test")
cls_test = arcpy.ia.ClassifyRaster(raster_test, str(used_ecd))
cls_test.save(classified_test)

class_outputs = pd.DataFrame([
    {"salida": "clasificacion_train", "ruta": classified_train},
    {"salida": "clasificacion_test", "ruta": classified_test},
    {"salida": "ecd_usado", "ruta": str(used_ecd)},
])
class_outputs.to_csv(OUT / "clase_05_salidas_clasificacion.csv", index=False, encoding="utf-8-sig")
class_outputs


## 8. Visualización rápida de rasters clasificados

**Qué:** exportamos una vista sencilla de los rasters clasificados.  
**Por qué:** la matriz de confusión no muestra patrones espaciales; el mapa permite detectar ruido, bordes y zonas dudosas.  
**Para qué:** complementar la evaluación numérica con lectura espacial.


In [ ]:
def raster_preview(raster_path, out_png, title):
    arr = arcpy.RasterToNumPyArray(raster_path, nodata_to_value=-9999)
    arr = np.where(arr == -9999, np.nan, arr)
    # reducir si es grande
    max_dim = max(arr.shape)
    step = max(1, int(math.ceil(max_dim / 1200)))
    arr2 = arr[::step, ::step]
    fig, ax = plt.subplots(figsize=(7,6))
    im = ax.imshow(arr2, cmap="viridis", interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
    cbar = plt.colorbar(im, ax=ax, shrink=0.75)
    cbar.set_label("Código de clase")
    plt.tight_layout()
    plt.savefig(out_png, dpi=170)
    plt.show()

raster_preview(classified_train, OUT / "clase_05_clasificacion_train.png", "Clasificación en zona de entrenamiento")
raster_preview(classified_test, OUT / "clase_05_clasificacion_test.png", "Clasificación en zona de prueba")


## 9. Evaluación con matriz de confusión sobre puntos de referencia

**Qué:** extraemos el valor clasificado en cada punto de entrenamiento y lo comparamos con su etiqueta real.  
**Por qué:** la clasificación debe evaluarse con datos etiquetados; una imagen visualmente plausible puede tener errores sistemáticos.  
**Para qué:** obtener exactitud, errores por clase y señales de sobreajuste.

Importante: esta matriz se calcula sobre la zona/puntos de entrenamiento. Es útil como control inicial, pero no reemplaza una validación independiente espacialmente separada.


In [ ]:
points_eval = str(OUT_GDB / "clase_05_puntos_eval")
arcpy.sa.ExtractValuesToPoints(training_fc, classified_train, points_eval, "NONE", "VALUE_ONLY")

eval_rows=[]
with arcpy.da.SearchCursor(points_eval, ["classname", "classvalue", "RASTERVALU"]) as cur:
    for classname, real, pred in cur:
        if real not in (1, 2) or pred not in (1, 2):
            raise ValueError("Clase desconocida o NoData; no truncar ni recodificar")
        eval_rows.append({"classname": classname, "real": int(real) if real is not None else None, "pred": int(pred) if pred is not None else None})
eval_df = pd.DataFrame(eval_rows)
if eval_df.isna().any().any() or not set(eval_df["pred"]).issubset({1, 2}):
    raise ValueError("Nulos/NoData o codigos desconocidos: revisar sin recodificar")

# Codigos SINCHI: 1=Bosque, 2=No Bosque; no sumar uno.
eval_df["pred_normalizada"] = eval_df["pred"]
normalizacion = "sin recodificacion; dominio 1/2 comprobado"

eval_df.to_csv(OUT / "clase_05_puntos_eval_predicciones.csv", index=False, encoding="utf-8-sig")
conf = pd.crosstab(eval_df["real"], eval_df["pred_normalizada"], rownames=["real"], colnames=["pred"])
conf.to_csv(OUT / "clase_05_matriz_confusion.csv", encoding="utf-8-sig")
accuracy = (eval_df["real"] == eval_df["pred_normalizada"]).mean()
metrics = pd.DataFrame([
    {"metrica": "exactitud_aparente_sobre_puntos", "valor": round(float(accuracy), 4), "n": len(eval_df)},
    {"metrica": "normalizacion_codigos", "valor": normalizacion, "n": len(eval_df)},
])
metrics.to_csv(OUT / "clase_05_metricas_clasificacion.csv", index=False, encoding="utf-8-sig")
conf, metrics


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
fig, ax = plt.subplots(figsize=(5.8,5))
im = ax.imshow(conf.values, cmap="YlGn")
ax.set_xticks(range(len(conf.columns)), labels=conf.columns)
ax.set_yticks(range(len(conf.index)), labels=conf.index)
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
ax.set_title("Matriz de confusión aparente")
for i in range(conf.shape[0]):
    for j in range(conf.shape[1]):
        ax.text(j, i, conf.values[i,j], ha="center", va="center", color="#111")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
fig_path = OUT / "clase_05_matriz_confusion.png"
plt.savefig(fig_path, dpi=170)
plt.show()
metrics


## 10. Interpretación: qué significa y qué no significa el resultado

**Qué aprendimos:**

1. El flujo bosque/no bosque se puede reproducir: referencia → muestras → clasificador `.ecd` → raster clasificado.
2. El balance de puntos ayuda a que ambas clases participen del entrenamiento.
3. La matriz calculada sobre puntos conocidos sirve como control de consistencia, pero puede ser optimista.
4. El mapa de prueba muestra transferencia espacial, pero sin verdad de terreno no permite medir exactitud real.

**Por qué importa:** en GeoIA, el valor no es solo ejecutar una herramienta. El valor está en documentar si el modelo puede generalizar y bajo qué condiciones.

**Para qué sirve:** este notebook deja una base para repetir la práctica, cambiar hiperparámetros, usar otra zona de validación y comparar modelos.


## 11. Limitaciones y siguientes pasos

- Separar una zona de validación que no participe en el entrenamiento.
- Evaluar exactitud con muestras independientes, no solo con los puntos usados para entrenar.
- Probar configuraciones con diferente número de árboles y profundidad.
- Revisar nubes, sombras y diferencias temporales entre referencia SINCHI e imagen Sentinel.
- Comparar clasificación balanceada frente a muestreo proporcional al área.
- Documentar explícitamente el orden `Band_1`–`Band_5` requerido por el `.ecd` antes de reutilizarlo.


# Extensión — reconstrucción completa y evaluación robusta

Esta extensión reconstruye la demo desde datos base y agrega pruebas metodológicas que no deben confundirse con un producto final. La intención es convertir la práctica en un laboratorio reproducible:

- generación de puntos aleatorios con ArcPy;
- extracción diagnóstica de variables disponibles a puntos;
- validación espacial independiente;
- comparación de hiperparámetros;
- otra partición de muestras;
- muestreo balanceado frente a proporcional al área;
- análisis de incertidumbre por consenso de modelos;
- validación con referencia externa disponible en el paquete local.


## 12. Funciones auxiliares para muestras, entrenamiento y evaluación

**Qué:** definimos funciones reutilizables para crear muestras, dividirlas, entrenar, clasificar y evaluar.  
**Por qué:** repetir manualmente los pasos aumenta errores y dificulta comparar escenarios.  
**Para qué:** que cada experimento cambie solo el criterio que se quiere comparar.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
import random
from collections import Counter

EXT = OUT / "extension"
EXT.mkdir(parents=True, exist_ok=True)
EXT_GDB = EXT / "clase_05_extension.gdb"
if not arcpy.Exists(str(EXT_GDB)):
    arcpy.management.CreateFileGDB(str(EXT), EXT_GDB.name)

arcpy.env.overwriteOutput = False
arcpy.env.workspace = str(EXT_GDB)

CLASS_MAP = {
    "BOSQUE": 1,
    "NO BOSQUE": 2,
}

def clean_name(text):
    return text.lower().replace(" ", "_").replace("/", "_").replace("-", "_")

def delete_if_exists(path):
    if arcpy.Exists(str(path)):
        raise FileExistsError(f"Se preserva {path}; reinicie con una carpeta nueva")

def class_area_table(source_fc=poly):
    rows=[]
    with arcpy.da.SearchCursor(source_fc, ["descripcio", "SHAPE@"]) as cur:
        for cls, area in cur:
            rows.append({"classname": str(cls).upper(), "area": area.getArea("GEODESIC", "SQUAREMETERS")})
    df=pd.DataFrame(rows).groupby("classname", as_index=False)["area"].sum()
    df["area_pct"]=(df["area"] / df["area"].sum()).round(4)
    return df

def make_class_points(source_fc, out_fc, counts_by_class, seed=17):
    """Crea puntos aleatorios por clase desde polígonos SINCHI y agrega classname/classvalue."""
    # La semilla Python no controla CreateRandomPoints.
    arcpy.env.randomGenerator = arcpy.CreateRandomValueGenerator(seed, "MERSENNE_TWISTER")
    delete_if_exists(out_fc)
    created=[]
    for classname, n in counts_by_class.items():
        lyr=f"lyr_{clean_name(classname)}_{seed}"
        with arcpy.da.SearchCursor(source_fc, ["descripcio"]) as rows:
            labels = {str(r[0]).upper(): r[0] for r in rows}
        if classname not in labels:
            raise ValueError(f"Clase ausente: {classname}")
        literal = str(labels[classname]).replace("'", "''")
        where = f"descripcio = '{literal}'"
        arcpy.management.MakeFeatureLayer(source_fc, lyr, where)
        tmp=str(EXT_GDB / f"tmp_pts_{clean_name(classname)}_{seed}_{n}")
        delete_if_exists(tmp)
        arcpy.management.CreateRandomPoints(str(EXT_GDB), Path(tmp).name, lyr, "", int(n), "", "POINT")
        arcpy.management.AddField(tmp, "classname", "TEXT", field_length=40)
        arcpy.management.AddField(tmp, "classvalue", "LONG")
        arcpy.management.CalculateField(tmp, "classname", repr(classname), "PYTHON3")
        arcpy.management.CalculateField(tmp, "classvalue", CLASS_MAP[classname], "PYTHON3")
        created.append(tmp)
        # Conservar capas y muestras intermedias.
    arcpy.management.Merge(created, out_fc)
    return out_fc

def add_xy_and_random(fc, seed=17):
    fields=[f.name for f in arcpy.ListFields(fc)]
    if "POINT_X" not in fields or "POINT_Y" not in fields:
        arcpy.management.CalculateGeometryAttributes(fc, [["POINT_X", "POINT_X"], ["POINT_Y", "POINT_Y"]], coordinate_system=arcpy.Describe(fc).spatialReference)
    fields=[f.name for f in arcpy.ListFields(fc)]
    if "rand_split" not in fields:
        arcpy.management.AddField(fc, "rand_split", "DOUBLE")
    if "sample_id" not in fields:
        arcpy.management.AddField(fc, "sample_id", "LONG")
        oid_name = arcpy.Describe(fc).OIDFieldName
        arcpy.management.CalculateField(fc, "sample_id", f"!{oid_name}!", "PYTHON3")
    oid=arcpy.Describe(fc).OIDFieldName
    with arcpy.da.UpdateCursor(fc, [oid, "rand_split"]) as cur:
        for oidv, _ in cur:
            rng=random.Random(seed + int(oidv)*1009)
            cur.updateRow([oidv, rng.random()])
    return fc

def split_points(fc, prefix, mode="random", train_ratio=0.7):
    add_xy_and_random(fc)
    train=str(EXT_GDB / f"{prefix}_train")
    valid=str(EXT_GDB / f"{prefix}_valid")
    delete_if_exists(train); delete_if_exists(valid)
    if mode == "random":
        train_where=f"rand_split <= {train_ratio}"
        valid_where=f"rand_split > {train_ratio}"
    elif mode == "spatial_x":
        xs=[]
        with arcpy.da.SearchCursor(fc, ["POINT_X"]) as cur:
            xs=[r[0] for r in cur if r[0] is not None]
        median_x=float(np.median(xs))
        train_where=f"POINT_X <= {median_x}"
        valid_where=f"POINT_X > {median_x}"
    else:
        raise ValueError(mode)
    arcpy.analysis.Select(fc, train, train_where)
    arcpy.analysis.Select(fc, valid, valid_where)
    return train, valid

def train_classify_eval(train_fc, valid_fc, label, trees=80, depth=30, samples=1000, classify_raster=raster_train):
    ecd=str(EXT / f"{label}_trees{trees}_depth{depth}.ecd")
    classified=str(EXT_GDB / f"{label}_cls_t{trees}_d{depth}")
    eval_fc=str(EXT_GDB / f"{label}_eval_t{trees}_d{depth}")
    for path in [ecd, classified, eval_fc]:
        delete_if_exists(path)
    arcpy.ia.TrainRandomTreesClassifier(classify_raster, train_fc, ecd, "#", trees, depth, samples, "#")
    result=arcpy.ia.ClassifyRaster(classify_raster, ecd)
    result.save(classified)
    arcpy.sa.ExtractValuesToPoints(valid_fc, classified, eval_fc, "NONE", "VALUE_ONLY")
    rows=[]
    with arcpy.da.SearchCursor(eval_fc, ["sample_id", "classname", "classvalue", "RASTERVALU"]) as cur:
        for sample_id, classname, real, pred in cur:
            if real not in (1, 2) or pred not in (1, 2):
                raise ValueError("Clase desconocida o NoData: no eliminar ni recodificar")
            rows.append({"sample_id": sample_id, "label": label, "classname": classname, "real": int(real), "pred_raw": int(pred), "trees": trees, "depth": depth})
    df=pd.DataFrame(rows)
    if df.empty:
        return df, classified, ecd
    df["pred"] = df["pred_raw"]
    norm="sin recodificacion; dominio 1/2 comprobado"
    df["correcta"] = df["real"] == df["pred"]
    df["normalizacion"] = norm
    return df, classified, ecd

def metrics_from_predictions(df, scenario):
    if df.empty:
        return {"escenario": scenario, "n": 0, "accuracy": np.nan}
    out={"escenario": scenario, "n": int(len(df)), "accuracy": round(float(df["correcta"].mean()), 4)}
    for cls, grp in df.groupby("real"):
        out[f"recall_clase_{int(cls)}"] = round(float((grp["pred"] == grp["real"]).mean()), 4)
    return out

area_df = class_area_table()
area_df.to_csv(EXT / "clase_05_area_clases_sinchi.csv", index=False, encoding="utf-8-sig")
area_df


## 13. Reconstrucción de puntos aleatorios con ArcPy

**Qué:** generamos dos muestras nuevas desde los polígonos SINCHI: una balanceada y otra proporcional al área.  
**Por qué:** la muestra balanceada fuerza igual peso por clase; la proporcional representa mejor la composición espacial, pero puede favorecer la clase dominante.  
**Para qué:** comparar cómo cambia el clasificador cuando cambia el diseño muestral.

Las áreas se calculan geodésicamente en metros cuadrados, no sobre SHAPE@AREA en grados. El muestreo proporcional conserva el mínimo original de 50 por clase y por ello es proporcional restringido. La semilla de Python no controla ArcGIS: se configura un generador ArcGIS explícito.


In [ ]:
# Muestra balanceada: misma cantidad por clase.
balanced_counts = {"BOSQUE": 500, "NO BOSQUE": 500}

# Muestra proporcional: total fijo repartido por área de clase.
total_prop = 1000
area_props = dict(zip(area_df["classname"], area_df["area_pct"]))
proportional_counts = {cls: max(50, int(round(total_prop * pct))) for cls, pct in area_props.items()}
# Ajuste simple para mantener el total cercano a 1000.
delta = total_prop - sum(proportional_counts.values())
if delta != 0:
    largest = max(proportional_counts, key=proportional_counts.get)
    proportional_counts[largest] += delta

balanced_fc = str(EXT_GDB / "muestra_balanceada_arcpy")
proportional_fc = str(EXT_GDB / "muestra_proporcional_arcpy")
dissolved_poly = str(SINCHI_GDB / "SINCHI_Guaviare_Dissolve")
make_class_points(dissolved_poly, balanced_fc, balanced_counts, seed=17)
make_class_points(dissolved_poly, proportional_fc, proportional_counts, seed=23)

sample_design = pd.DataFrame([
    {"muestra": "balanceada", "clase": k, "puntos": v} for k, v in balanced_counts.items()
] + [
    {"muestra": "proporcional_area", "clase": k, "puntos": v} for k, v in proportional_counts.items()
])
sample_design.to_csv(EXT / "clase_05_diseno_muestras.csv", index=False, encoding="utf-8-sig")
sample_design


## 14. Extracción diagnóstica de NDVI a puntos nuevos

**Qué:** extraemos el raster `NDVI_20230823_10m` a las muestras generadas como variable auxiliar de diagnóstico.  
**Por qué:** `NDVI_20230823_10m` existe en `SINCHI.gdb`, pero no debe confundirse con los nombres `Band_1`–`Band_5` del raster compuesto ni con una afirmación de que toda la clasificación dependa del NDVI.  
**Para qué:** revisar separabilidad de las muestras con una variable interpretable, manteniendo la clasificación basada en el raster compuesto completo.


In [ ]:
def extract_values_for_audit(fc, out_csv, label):
    fc_copy=str(EXT_GDB / f"{label}_audit")
    delete_if_exists(fc_copy)
    arcpy.management.CopyFeatures(fc, fc_copy)
    # NDVI se extrae como variable auxiliar de diagnóstico; no sustituye al raster compuesto Band_1-Band_5.
    arcpy.sa.ExtractMultiValuesToPoints(fc_copy, [[str(SINCHI_GDB / "NDVI_20230823_10m"), "NDVI_ext"]], "NONE")
    rows=[]
    with arcpy.da.SearchCursor(fc_copy, ["classname", "classvalue", "NDVI_ext", "SHAPE@XY"]) as cur:
        for classname, classvalue, ndvi, xy in cur:
            rows.append({"muestra": label, "classname": classname, "classvalue": classvalue, "NDVI_ext": ndvi, "x": xy[0], "y": xy[1]})
    df=pd.DataFrame(rows)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    return df

balanced_audit = extract_values_for_audit(balanced_fc, EXT / "clase_05_muestra_balanceada_ndvi.csv", "balanceada")
proportional_audit = extract_values_for_audit(proportional_fc, EXT / "clase_05_muestra_proporcional_ndvi.csv", "proporcional")

fig, ax = plt.subplots(figsize=(8,4))
for label, df in [("balanceada", balanced_audit), ("proporcional", proportional_audit)]:
    means = df.groupby("classname")["NDVI_ext"].mean()
    ax.plot(means.index, means.values, marker="o", label=label)
ax.set_title("NDVI medio por clase en muestras reconstruidas")
ax.set_ylabel("NDVI")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(EXT / "clase_05_ndvi_muestras_reconstruidas.png", dpi=160)
plt.show()
pd.concat([balanced_audit.assign(muestra="balanceada"), proportional_audit.assign(muestra="proporcional")]).groupby(["muestra","classname"])["NDVI_ext"].agg(["count","mean","std","min","max"]).round(4)


## 15. Validación espacial independiente

**Qué:** dividimos la muestra balanceada por posición X: entrenamiento al oeste y validación al este.  
**Por qué:** una partición aleatoria mezcla vecinos espaciales y puede inflar la exactitud. Una separación espacial prueba mejor la capacidad de generalizar.  
**Para qué:** detectar sobreajuste espacial o dependencia excesiva del contexto local.

Separar por mediana X no garantiza independencia: no hay buffer y puntos a ambos lados pueden ser vecinos del mismo polígono. Interprete este contraste como partición espacial exploratoria.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
sp_train, sp_valid = split_points(balanced_fc, "balanceada_spatial_x", mode="spatial_x")
sp_eval, sp_cls, sp_ecd = train_classify_eval(sp_train, sp_valid, "validacion_espacial", trees=80, depth=30, samples=500)
sp_metrics = pd.DataFrame([metrics_from_predictions(sp_eval, "validacion_espacial_x")])
sp_conf = pd.crosstab(sp_eval["real"], sp_eval["pred"], rownames=["real"], colnames=["pred"])
sp_eval.to_csv(EXT / "clase_05_validacion_espacial_predicciones.csv", index=False, encoding="utf-8-sig")
sp_conf.to_csv(EXT / "clase_05_validacion_espacial_matriz.csv", encoding="utf-8-sig")
sp_metrics.to_csv(EXT / "clase_05_validacion_espacial_metricas.csv", index=False, encoding="utf-8-sig")
sp_conf, sp_metrics


## 16. Otra partición de muestras: validación aleatoria 70/30

**Qué:** repetimos el experimento con una partición aleatoria 70/30.  
**Por qué:** permite comparar el desempeño aparente cuando entrenamiento y validación están mezclados espacialmente.  
**Para qué:** contrastar validación aleatoria frente a validación espacial.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
rnd_train, rnd_valid = split_points(balanced_fc, "balanceada_random_70_30", mode="random", train_ratio=0.7)
rnd_eval, rnd_cls, rnd_ecd = train_classify_eval(rnd_train, rnd_valid, "validacion_random", trees=80, depth=30, samples=500)
rnd_metrics = pd.DataFrame([metrics_from_predictions(rnd_eval, "validacion_random_70_30")])
rnd_conf = pd.crosstab(rnd_eval["real"], rnd_eval["pred"], rownames=["real"], colnames=["pred"])
rnd_eval.to_csv(EXT / "clase_05_validacion_random_predicciones.csv", index=False, encoding="utf-8-sig")
rnd_conf.to_csv(EXT / "clase_05_validacion_random_matriz.csv", encoding="utf-8-sig")
rnd_metrics.to_csv(EXT / "clase_05_validacion_random_metricas.csv", index=False, encoding="utf-8-sig")
pd.concat([sp_metrics, rnd_metrics], ignore_index=True)


## 17. Comparación de hiperparámetros

**Qué:** entrenamos varios modelos cambiando número de árboles y profundidad máxima.  
**Por qué:** más complejidad no siempre mejora la generalización; puede aumentar sobreajuste o costo.  
**Para qué:** observar sensibilidad del resultado a decisiones del modelo.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
hyperparams = [
    {"trees": 30, "depth": 10},
    {"trees": 80, "depth": 20},
    {"trees": 150, "depth": 30},
]
hyper_evals=[]
hyper_metrics=[]
hyper_outputs=[]
for hp in hyperparams:
    label=f"hp_t{hp['trees']}_d{hp['depth']}"
    df, cls_path, ecd_path = train_classify_eval(rnd_train, rnd_valid, label, trees=hp["trees"], depth=hp["depth"], samples=500)
    hyper_evals.append(df.assign(modelo=label))
    hyper_metrics.append(metrics_from_predictions(df, label))
    hyper_outputs.append({"modelo": label, "raster": cls_path, "ecd": ecd_path})

hyper_eval_df = pd.concat(hyper_evals, ignore_index=True)
hyper_metrics_df = pd.DataFrame(hyper_metrics)
hyper_outputs_df = pd.DataFrame(hyper_outputs)
hyper_eval_df.to_csv(EXT / "clase_05_hiperparametros_predicciones.csv", index=False, encoding="utf-8-sig")
hyper_metrics_df.to_csv(EXT / "clase_05_hiperparametros_metricas.csv", index=False, encoding="utf-8-sig")
hyper_outputs_df.to_csv(EXT / "clase_05_hiperparametros_salidas.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(8,4))
ax.bar(hyper_metrics_df["escenario"], hyper_metrics_df["accuracy"], color="#1976d2")
ax.set_ylim(0, 1)
ax.set_ylabel("Exactitud en validación 70/30")
ax.set_title("Comparación de hiperparámetros Random Trees")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(EXT / "clase_05_comparacion_hiperparametros.png", dpi=160)
plt.show()
hyper_metrics_df


## 18. Muestreo balanceado vs proporcional al área

**Qué:** entrenamos dos modelos con el mismo diseño de partición, pero distinta estrategia de muestreo.  
**Por qué:** el balance mejora aprendizaje de clases minoritarias, mientras que el muestreo proporcional representa mejor el paisaje pero puede sesgar la predicción.  
**Para qué:** decidir si el objetivo es maximizar exactitud global o proteger el desempeño por clase.


In [ ]:
# Revisar entradas y proposito antes de ejecutar esta etapa.
prop_train, prop_valid = split_points(proportional_fc, "proporcional_random_70_30", mode="random", train_ratio=0.7)
bal_eval, _, _ = train_classify_eval(rnd_train, rnd_valid, "muestreo_balanceado", trees=80, depth=20, samples=500)
prop_eval, _, _ = train_classify_eval(prop_train, prop_valid, "muestreo_proporcional", trees=80, depth=20, samples=800)

sampling_metrics = pd.DataFrame([
    metrics_from_predictions(bal_eval, "muestreo_balanceado"),
    metrics_from_predictions(prop_eval, "muestreo_proporcional_area"),
])
sampling_metrics.to_csv(EXT / "clase_05_muestreo_balanceado_vs_proporcional_metricas.csv", index=False, encoding="utf-8-sig")
pd.concat([bal_eval.assign(muestreo="balanceado"), prop_eval.assign(muestreo="proporcional")], ignore_index=True).to_csv(
    EXT / "clase_05_muestreo_balanceado_vs_proporcional_predicciones.csv", index=False, encoding="utf-8-sig"
)

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(sampling_metrics["escenario"], sampling_metrics["accuracy"], color=["#2e7d32", "#f9a825"])
ax.set_ylim(0, 1)
ax.set_ylabel("Exactitud")
ax.set_title("Efecto del diseño de muestreo")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(EXT / "clase_05_muestreo_balanceado_vs_proporcional.png", dpi=160)
plt.show()
sampling_metrics


## 19. Incertidumbre por consenso de modelos

**Qué:** usamos los modelos de hiperparámetros como un pequeño ensamble y medimos acuerdo de voto en los puntos de validación.  
**Por qué:** si distintos modelos predicen clases diferentes para el mismo punto, la clasificación es menos confiable.  
**Para qué:** producir una lectura simple de incertidumbre sin requerir probabilidades internas del clasificador.


In [ ]:
# Alinear por identificador persistente del punto, nunca por posicion de fila.
unc = hyper_eval_df.copy()
if unc.duplicated(["modelo", "sample_id"]).any():
    raise ValueError("Identificadores repetidos; no deduplicar")
wide = unc.pivot(index=["sample_id", "real", "classname"], columns="modelo", values="pred").reset_index()
if wide.isna().any().any():
    raise ValueError("Conjuntos de puntos distintos; no calcular consenso parcial")
model_cols = [c for c in wide.columns if str(c).startswith("hp_")]

def vote_stats(row):
    votes=[row[c] for c in model_cols if pd.notna(row[c])]
    counts=Counter(votes)
    majority_class, majority_n = counts.most_common(1)[0]
    confidence = majority_n / len(votes)
    probs=np.array([v/len(votes) for v in counts.values()], dtype=float)
    entropy=float(-(probs*np.log2(probs)).sum()) if len(probs)>1 else 0.0
    return pd.Series({"voto_mayoritario": majority_class, "confianza_voto": confidence, "entropia_voto": entropy})

vote_df = pd.concat([wide, wide.apply(vote_stats, axis=1)], axis=1)
vote_df["correcta_consenso"] = vote_df["real"] == vote_df["voto_mayoritario"]
vote_summary = pd.DataFrame([{
    "n": len(vote_df),
    "accuracy_consenso": round(float(vote_df["correcta_consenso"].mean()), 4),
    "confianza_media": round(float(vote_df["confianza_voto"].mean()), 4),
    "pct_baja_confianza_lt_1": round(float((vote_df["confianza_voto"] < 1).mean()), 4),
}])
vote_df.to_csv(EXT / "clase_05_incertidumbre_consenso_puntos.csv", index=False, encoding="utf-8-sig")
vote_summary.to_csv(EXT / "clase_05_incertidumbre_consenso_resumen.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7,4))
ax.hist(vote_df["confianza_voto"], bins=[0.3,0.5,0.67,0.84,1.01], color="#6a1b9a", edgecolor="white")
ax.set_title("Incertidumbre por consenso de modelos")
ax.set_xlabel("Confianza de voto")
ax.set_ylabel("Puntos de validación")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(EXT / "clase_05_incertidumbre_consenso.png", dpi=160)
plt.show()
vote_summary


## 20. Validación con referencia externa disponible

**Qué:** buscamos datos externos al flujo de entrenamiento dentro del paquete local. No hay una segunda fuente independiente distinta a SINCHI; por eso se crea una muestra de referencia separada desde los polígonos SINCHI y se evalúa el mejor modelo contra ella.  
**Por qué:** una validación externa estricta requiere otra fuente o campaña independiente. Si no existe, se debe declarar la limitación en vez de simular certeza.  
**Para qué:** dejar preparado el patrón de evaluación externa y, mientras tanto, producir una validación proxy contra referencia no usada en entrenamiento.


In [ ]:
# Validación externa/proxy: muestra nueva de la misma referencia, no independiente desde polígonos de referencia SINCHI.
external_counts = {"BOSQUE": 300, "NO BOSQUE": 300}
external_fc = str(EXT_GDB / "muestra_referencia_externa_proxy")
make_class_points(dissolved_poly, external_fc, external_counts, seed=99)

# Mejor modelo por exactitud de la comparación de hiperparámetros.
best_row = hyper_metrics_df.sort_values("accuracy", ascending=False).iloc[0]
best_label = best_row["escenario"]
best_output = hyper_outputs_df.loc[hyper_outputs_df["modelo"] == best_label].iloc[0]
best_raster = best_output["raster"]

external_eval_fc = str(EXT_GDB / "validacion_externa_proxy_eval")
delete_if_exists(external_eval_fc)
arcpy.sa.ExtractValuesToPoints(external_fc, best_raster, external_eval_fc, "NONE", "VALUE_ONLY")
rows=[]
with arcpy.da.SearchCursor(external_eval_fc, ["classname", "classvalue", "RASTERVALU"]) as cur:
    for classname, real, pred in cur:
        if real not in (1, 2) or pred not in (1, 2):
            raise ValueError("Clase desconocida o NoData en referencia proxy")
        rows.append({"classname": classname, "real": int(real), "pred_raw": int(pred), "modelo": best_label})
ext_df=pd.DataFrame(rows)
if not ext_df.empty:
    ext_df["pred"] = ext_df["pred_raw"]
    norm = "sin recodificacion; dominio 1/2 comprobado"
    ext_df["correcta"] = ext_df["real"] == ext_df["pred"]
    ext_metrics = pd.DataFrame([metrics_from_predictions(ext_df, "validacion_externa_proxy_sinchi") | {"modelo": best_label, "normalizacion": norm}])
    ext_conf = pd.crosstab(ext_df["real"], ext_df["pred"], rownames=["real"], colnames=["pred"])
else:
    ext_metrics = pd.DataFrame([{"escenario":"validacion_externa_proxy_sinchi", "n":0, "accuracy":np.nan, "modelo":best_label}])
    ext_conf = pd.DataFrame()

ext_df.to_csv(EXT / "clase_05_validacion_externa_proxy_predicciones.csv", index=False, encoding="utf-8-sig")
ext_metrics.to_csv(EXT / "clase_05_validacion_externa_proxy_metricas.csv", index=False, encoding="utf-8-sig")
ext_conf.to_csv(EXT / "clase_05_validacion_externa_proxy_matriz.csv", encoding="utf-8-sig")
ext_conf, ext_metrics


## 21. Síntesis de la extensión

**Qué muestran los experimentos:**

- La demo puede reconstruirse desde polígonos base mediante puntos aleatorios generados con ArcPy.
- La validación espacial suele ser más exigente que una partición aleatoria porque separa zonas, no solo filas.
- Los hiperparámetros cambian el comportamiento del modelo; por eso deben reportarse y compararse.
- El muestreo balanceado y el proporcional responden a objetivos distintos: equidad por clase frente a representación del paisaje.
- La incertidumbre puede aproximarse mediante desacuerdo entre modelos cuando no se dispone de probabilidades internas.
- La validación externa estricta queda pendiente si no se incorpora una fuente independiente adicional; la muestra proxy SINCHI solo valida contra referencia separada del entrenamiento.

**Para un producto final:** usar una validación espacial independiente real, datos de referencia externos y revisión de incertidumbre antes de tomar decisiones operativas.


## Procedencia y estado
Fuente: notebook Clase 17, 01/07/2026; docente fuente José Sebastián Gómez Romero. Destino: Clase 05, 22/09/2026, Fabian Cetina. Se conservan las 46 celdas originales en orden y sus ejercicios; se añade configuración y este registro. Las salidas históricas siguen en el original de solo lectura; no se atribuyen a esta adaptación. Las métricas afectadas por interpolación, recodificación y consenso posicional requieren recalcularse.

P01 → este notebook → `datos/clase_05/SINCHI.gdb` → `salidas_clase_05/practica_01/ejecucion_*`. No se duplicó la GDB. Coiba se discute desde [P01 de Clase 04](Clase%2004%20-%20Practica%2001%20-%20Random%20Forest%20geoespacial.ipynb).

Correcciones respaldadas por documentación oficial instalada Esri ArcGIS Pro 3.6.2 (consulta 2026-09-22): `arcpy.sa.Functions.ExtractValuesToPoints`, interpolate_values: NONE usa la celda, INTERPOLATE aplica bilineal; `Geometry.getArea`, GEODESIC/SQUAREMETERS; `arcpy.CreateRandomValueGenerator`, seed/distribution; `arcpy.charts.Bar`, x/aggregation/dataSource/exportToSVG. Son los pasajes locales de la API, no una consulta web nueva. Se conserva Matplotlib como complemento original y se añade barra nativa ArcGIS; mapas/vistas ArcGIS y comprobación de renderizado siguen pendientes.

No se ejecutaron análisis ni modelos. Entorno comunicado por inspección previa: Pro 3.6.2, ArcInfo; disponibilidad de extensiones no prueba préstamo. Inspección previa: 513 polígonos (90 bosque, 423 no bosque), 2 clases disueltas, 2.000 puntos balanceados; vectores EPSG:4170, ráster EPSG:32618 a 10 m. Band_1–Band_5 son posiciones, no identidad espectral. La transferencia a Raster_test se detiene hasta verificar composición.

Video 20260701_225550UTC, duración 2:01:45: coordinador confirmó reproductor y cobertura parcial en la sesión. Pendientes: cobertura docente pertinente completa, seis diapositivas válidas, identidad de bandas, ejecución autorizada y revisión académica. La validación proxy usa la misma referencia SINCHI; no es validación externa independiente.
